In [ ]:
import pandas as pd

# 1. Import des données

In [ ]:
df_irve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435'
)

print(f"Shape : {df_irve.shape}")
df_irve.sample(5)

Ce premier jeu de données est la base nationale des IRVE (Infrastructures de Recharge pour Véhicules Électriques).
Cette table comporte 212234 lignes pour 52 variables.

In [ ]:
df_ve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/90e0d717-deda-4bdc-9987-f82faac5bc93',
 encoding='latin-1'
)

print(f"Shape : {df_ve.shape}")
df_ve.sample(5)

Ce second jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge. Il contient l'historique des immatriculations pour chaque commune à différentes dates.
Cette table comporte 703545 lignes pour 8 variables. 

In [ ]:
df_revenus = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',sep=';'
)

print(f"Shape : {df_revenus.shape}")
df_revenus.sample(5)

Ce dernier jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge.
Cette table comporte 34926 lignes pour 57 variables.

# 2. Jointure des tables

## 2.1 Choix de la variable de jointure

Les 3 jeu de données contiennent une variable indiquant le code géographique mais sous 3 noms différents :
- `code_insee_commune` pour df_irve
- `CODGEO` pour df_ve
- `Code géographique` pour df_revenus

La jointure sera faite sur ces variables.

## 2.2 Préparation de la variable de jointure

Le diagnostic met en évidence plusieurs points d’attention :
- Présence de valeurs manquantes dans `code_insee_commune` (df_irve)  
- Présence de longueurs de codes incohérentes :
  - `code_insee_commune` (df_irve) contient des codes de longueur 3, 6 et 7
  - `CODGEO` (df_ve) contient des codes de longueur 4
- Les variables de jointure ne sont pas uniques :
  - `code_insee_commune` (df_irve)
  - `CODGEO` (df_ve)

Les clés de jointure présentent des problèmes de qualité et de format.  
Il est nécessaire de les nettoyer et les uniformiser (type, format, gestion des valeurs manquantes, unicité) avant d’effectuer la jointure.

### 2.2.1 Unification des codes

In [ ]:
from src.preparation_data import nettoyer_code_insee

In [ ]:
df_irve["code_insee_commune"] = df_irve["code_insee_commune"].apply(nettoyer_code_insee)
df_ve["CODGEO"] = df_ve["CODGEO"].apply(nettoyer_code_insee)
df_revenus["Code géographique"] = df_revenus["Code géographique"].apply(nettoyer_code_insee)

Les codes insee sont maintenant au même format dans les 3 bases de données : un code à 5 caractères.

### 2.2.2 Compléter les valeurs manquantes

In [ ]:
from src.preparation_data import creer_gdf_irve, charger_communes, joindre_communes, ajouter_codes_geo

In [ ]:
gdf_irve = creer_gdf_irve(df_irve, "consolidated_longitude", "consolidated_latitude")
gdf_communes = charger_communes()
gdf_result = joindre_communes(gdf_irve, gdf_communes)
df_irve = ajouter_codes_geo(df_irve, gdf_result)

Nous avons maintenant 2 colonnes concernant le code géographique dans la base de données IRVE :
- `code_insee_commune` : la variable initiale
- `code_geo_total` : la variable recalculant tous les codes géographiques à partir de la latitude et la longitude

In [ ]:
from src.preparation_data import compter_valeurs_manquantes, compter_uniques

In [ ]:
colonnes_irve = ["code_insee_commune", "code_geo_total"]

print("Valeurs manquantes :")
print(compter_valeurs_manquantes(df_irve, colonnes_irve))

print("\nValeurs uniques :")
print(compter_uniques(df_irve, colonnes_irve))

Cette manipulation permet de passer de 57919 valeurs manquantes à 1768. De plus les codes recalculés sont plus fiables que ceux initiaux.

## 2.3 Jointure

Réfléchissons aux variables à garder... Lesquelles pourraient être intéressante ?

In [ ]:
list(df_irve.columns)

In [ ]:
list(df_ve.columns)

In [ ]:
list(df_revenus.columns)

### 2.3.1 Une ligne par code géo

Il faut réfléchir à la façon dont on veut agréger df_irve et quelles variables on souhaite garder.

##### df_irve

Ici, l'objectif est de mesurer l'offre de recharge par commune. Comme il peut y avoir plusieurs points de recharge par commune, il faut agréger.

**Variables à garder :** nbre_pdc, puissance_nominale, gratuit, paiement_cb, condition_acces.

**Traitement (Aggregation) :**
- nb_stations : Compter le nombre de lignes (ou id_station_local uniques).
- total_pdc : Somme de nbre_pdc (nombre total de points de charge).
- puissance_max : Maximum de puissance_nominale (pour identifier les communes avec recharge ultra-rapide).
- part_gratuit : Pourcentage de points de charge où gratuit est vrai. C'est un excellent indicateur pour l'attractivité d'une commune.
- acces_public : Compter les stations dont la condition_acces est "Accès libre".

In [ ]:
from src.nouvelles_variables import creer_features_irve

mettre ça ailleurs

In [ ]:
def pipeline_irve(df):
    """
    Pipeline complet de préparation et feature engineering IRVE
    """
    df = df.copy()

    # =========================
    # 1. FILTRAGE VARIABLES
    # =========================
    var_interet = [
        'nom_operateur',
        'code_geo_total',
        'implantation_station',
        'nbre_pdc',
        'puissance_nominale',
        'prise_type_ef',
        'prise_type_2',
        'prise_type_combo_ccs',
        'prise_type_chademo',
        'gratuit',
        'paiement_acte',
        'paiement_cb',
        'paiement_autre',
        'tarification',
        'condition_acces',
        'reservation',
        'horaires',
        'created_at']

    df = df[var_interet]

    # =========================
    # 2. TYPES BOOLEAN
    # =========================
    cols_bool = [
        'prise_type_ef',
        'prise_type_2',
        'prise_type_combo_ccs',
        'prise_type_chademo',
        'prise_type_autre',
        'cable_t2_attache',
        'gratuit',
        'paiement_acte',
        'paiement_cb',
        'paiement_autre',
        'reservation'
    ]

    mapping = {
        'true': True,
        'false': False,
        '1': True,
        '0': False
    }

    for col in cols_bool:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .map(mapping)
            .astype("boolean")
        )

    # =========================
    # 3. DATES
    # =========================
    df['created_at'] = pd.to_datetime(df['created_at'])
    df['annee'] = df['created_at'].dt.year

    # =========================
    # 4. STRING
    # =========================
    cols_str = [
        'nom_operateur',
        'implantation_station',
        'tarification',
        'condition_acces',
        'horaires'
    ]

    for col in cols_str:
        df[col] = df[col].astype("string")

    # catégories
    df['implantation_station'] = df['implantation_station'].astype('category')
    df['condition_acces'] = df['condition_acces'].astype('category')

    return df

In [ ]:
df_filtre = pipeline_irve(df_irve)
df_filtre.sample(5)

In [ ]:
nv_var = creer_features_irve(df_filtre)

In [ ]:
nv_var.sample(5)

##### df_ve

Pour que l'analyse soit cohérente avec les données de bornes (IRVE) et de l'INSEE (qui sont des "clichés" à un instant T), il faut isoler la situation la plus récente pour chaque commune afin d'avoir une unique ligne par code insee.

In [ ]:
df_ve["DATE_ARRETE"].dtype

In [ ]:
ve = df_ve.copy()

# Conversion en format date pour être sûr du tri
ve['DATE_ARRETE'] = pd.to_datetime(ve['DATE_ARRETE'])

# Tri par commune puis par date (de la plus ancienne à la plus récente)
ve = ve.sort_values(['CODGEO', 'DATE_ARRETE'])

# garde que la DERNIÈRE ligne pour chaque CODGEO (la plus récente)
df_ve_latest = ve.drop_duplicates(subset='CODGEO', keep='last')

df_ve_latest

**Variables à garder :** NB_VP_RECHARGEABLES_EL, NB_VP.

**Traitement :**
taux_equipement_ve : Créer un ratio (NB_VP_RECHARGEABLES_EL / NB_VP). C'est beaucoup plus pertinent que le chiffre brut, car cela normalise par la taille de la commune.

##### df_revenus

In [ ]:
# On garde le code commune et le revenu médian (souvent colonne 'MED21' pour 2021)
df_revenus_final = df_revenus[['COM', 'MED21']].copy()
df_revenus_final.rename(columns={'COM': 'code_commune_insee', 'MED21': 'revenu_median'}, inplace=True)

**Variables à garder :**
[DISP] Médiane (€), [DISP] Part des revenus d’activité (%), [DISP] Iice de Gini.

Intérêt : La médiane du revenu est indispensable pour vérifier si l'adoption du VE est corrélée à la richesse de la commune. L'indice de Gini peut montrer si les communes très inégalitaires ont un comportement différent.

On peut en choisir d'autres. D'ailleurs on doit choisir uniquement les variables de df_irve avant la jointure car pour les 2 autres tables on a déjà une ligne par commune donc pas besoin de faire d'agrégation. Ainsi on peut toutes les prendre et choisir après.

Après le choix des variables : 
0. variable cible = taux de VE
1. faire des analyses descriptives : 
- automatiser avec une fonction
- matrice de corrélation (heatmap)
2. modélisation :
simple regression ou plus avace (random forest...)